In [25]:
import pandas as pd
print("Funcionando!")
print(pd.__version__)

Funcionando!
3.0.5


In [ ]:
books = pd.read_csv('data/kaggle/books.csv')
tags = pd.read_csv('data/kaggle/tags.csv')
book_tags = pd.read_csv('data/kaggle/book_tags.csv')

In [27]:
book_tags = book_tags.merge(tags, on='tag_id') #une as duas tabelas onde o tag_id for igual
book_tags.head()

,goodreads_book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult


In [28]:
generos_validos = [
    'fiction', 'fantasy', 'young-adult', 'romance', 'mystery', 'thriller',
    'horror', 'science-fiction', 'sci-fi', 'classics', 'historical-fiction',
    'nonfiction', 'non-fiction', 'biography', 'memoir', 'poetry',
    'contemporary', 'adventure', 'dystopia', 'crime', 'humor',
    'graphic-novels', 'self-help', 'philosophy', 'psychology',
    'history', 'science', 'business', 'true-crime', 'short-stories'
]

book_tags_filtrado = book_tags[book_tags['tag_name'].isin(generos_validos)]
print(book_tags_filtrado.shape)
book_tags_filtrado['tag_name'].value_counts()

(66397, 4)


tag_name
fiction               9097
contemporary          5287
fantasy               4259
romance               4251
mystery               3686
adventure             3661
young-adult           3630
classics              2785
historical-fiction    2590
thriller              2522
sci-fi                2227
science-fiction       2222
humor                 2161
history               2138
non-fiction           2128
crime                 2083
nonfiction            1833
horror                1372
science               1239
biography             1109
philosophy            1055
memoir                 905
psychology             810
dystopia               718
short-stories          711
self-help              579
graphic-novels         491
poetry                 377
business               377
true-crime              94
Name: count, dtype: int64

In [29]:
mapa_unificacao = {
    'sci-fi': 'science-fiction',
    'nonfiction': 'non-fiction',
}

book_tags_filtrado = book_tags_filtrado.copy()
book_tags_filtrado['tag_name'] = book_tags_filtrado['tag_name'].replace(mapa_unificacao)

book_tags_filtrado['tag_name'].value_counts()

tag_name
fiction               9097
contemporary          5287
science-fiction       4449
fantasy               4259
romance               4251
non-fiction           3961
mystery               3686
adventure             3661
young-adult           3630
classics              2785
historical-fiction    2590
thriller              2522
humor                 2161
history               2138
crime                 2083
horror                1372
science               1239
biography             1109
philosophy            1055
memoir                 905
psychology             810
dystopia               718
short-stories          711
self-help              579
graphic-novels         491
poetry                 377
business               377
true-crime              94
Name: count, dtype: int64

In [30]:
book_tags_agg = (
    book_tags_filtrado.sort_values('count', ascending=False)
    .groupby('goodreads_book_id')
    .head(5)  # top 5 tags mais votadas por livro, dentro das válidas
    .groupby('goodreads_book_id')['tag_name']
    .apply(lambda tags: ' '.join(tags))
    .reset_index()
)

book_tags_agg.columns = ['goodreads_book_id', 'genero_texto']
print(book_tags_agg.shape)
book_tags_agg.head(10)

(9999, 2)


,goodreads_book_id,genero_texto
0,1,fantasy young-adult fiction adventure classics
1,2,fantasy fiction young-adult mystery romance
2,3,fantasy young-adult fiction adventure classics
3,5,fantasy young-adult fiction adventure classics
4,6,fantasy young-adult fiction adventure classics
5,8,fantasy young-adult fiction adventure classics
6,10,fantasy fiction young-adult adventure classics
7,11,science-fiction science-fiction fiction humor ...
8,13,science-fiction science-fiction fiction humor ...
9,21,history non-fiction non-fiction science humor


In [31]:
# Soma o count de tags que viraram iguais após a unificação (ex: sci-fi + science-fiction)
book_tags_somado = (
    book_tags_filtrado
    .groupby(['goodreads_book_id', 'tag_name'], as_index=False)['count']
    .sum()
)

# Agrega top 5 tags por livro, sem duplicar nomes
book_tags_agg = (
    book_tags_somado.sort_values('count', ascending=False)
    .groupby('goodreads_book_id')
    .head(5)
    .groupby('goodreads_book_id')['tag_name']
    .apply(lambda tags: ' '.join(tags))
    .reset_index()
)

book_tags_agg.columns = ['goodreads_book_id', 'genero_texto']
print(book_tags_agg.shape)
book_tags_agg.head(10)

(9999, 2)


,goodreads_book_id,genero_texto
0,1,fantasy young-adult fiction adventure classics
1,2,fantasy fiction young-adult mystery romance
2,3,fantasy young-adult fiction adventure classics
3,5,fantasy young-adult fiction adventure classics
4,6,fantasy young-adult fiction adventure classics
5,8,fantasy young-adult fiction adventure classics
6,10,fantasy fiction young-adult adventure classics
7,11,science-fiction fiction humor fantasy classics
8,13,science-fiction fiction humor fantasy classics
9,21,history non-fiction science humor philosophy


In [32]:
books_final = books.merge(
    book_tags_agg,
    left_on='book_id',
    right_on='goodreads_book_id',
    how='left'
)

print("Total de livros:", books_final.shape[0])
print("Livros sem genero_texto:", books_final['genero_texto'].isna().sum())

books_final[['title', 'authors', 'genero_texto']].head(10)

Total de livros: 10000
Livros sem genero_texto: 1


,title,authors,genero_texto
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,young-adult science-fiction fiction dystopia f...
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",fantasy young-adult fiction adventure classics
2,"Twilight (Twilight, #1)",Stephenie Meyer,young-adult fantasy fiction contemporary scien...
3,To Kill a Mockingbird,Harper Lee,classics historical-fiction young-adult fictio...
4,The Great Gatsby,F. Scott Fitzgerald,classics fiction historical-fiction romance yo...
5,The Fault in Our Stars,John Green,young-adult fiction romance contemporary humor
6,The Hobbit,J.R.R. Tolkien,fantasy classics fiction adventure young-adult
7,The Catcher in the Rye,J.D. Salinger,classics fiction young-adult contemporary hist...
8,"Angels & Demons (Robert Langdon, #1)",Dan Brown,fiction mystery thriller adventure historical-...
9,Pride and Prejudice,Jane Austen,classics fiction romance historical-fiction hi...


In [33]:
books_final = books_final.dropna(subset=['genero_texto']).reset_index(drop=True)
print("Livros restantes:", books_final.shape[0])

Livros restantes: 9999


In [34]:
books_final.to_csv('data/books_final.csv', index=False)